# درس ۱۰ - عامل‌های هوش مصنوعی در تولید

در این درس شما با **الگوهای تولید** برای عامل‌های هوش مصنوعی با استفاده از Microsoft Agent Framework (پایتون) آشنا می‌شوید. موضوعات شامل:

- **قابلیت مشاهده** — افزودن زمان‌بندی و ثبت رویدادها به تعاملات عامل
- **ارزیابی** — استفاده از یک عامل ارزیاب برای امتیازدهی کیفیت پاسخ‌ها
- **مدیریت هزینه** — استراتژی‌هایی برای بهینه‌سازی توکن و انتخاب مدل

سناریو، یک **عامل مسافرتی** است که به کاربران در برنامه‌ریزی سفرها کمک می‌کند، با لایه‌ای از نظارت و ارزیابی در بالا.


## نصب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ملاحظات تولید

منتقل کردن عامل‌های هوش مصنوعی از نمونه اولیه به تولید نیازمند توجه دقیق به سه رکن است:

1. **قابلیت مشاهده** — شما نیاز به دید دقیق نسبت به کاری که عامل انجام می‌دهد، مدت زمانی که می‌برد و ابزارهایی که فراخوانی می‌کند دارید. بدون ردیابی و ثبت وقایع، عیب‌یابی مشکلات در تولید تقریبا غیرممکن است.

2. **ارزیابی** — بررسی‌های خودکار کیفیت تضمین می‌کند که پاسخ‌های عامل در طول زمان دقیق، کامل و مفید باقی بماند. یک عامل ارزیاب می‌تواند پاسخ‌ها را بر اساس معیارهای تعریف شده امتیاز دهد.

3. **مدیریت هزینه** — استفاده از توکن به طور مستقیم بر هزینه تأثیر می‌گذارد. راهبردهایی مانند بهینه‌سازی درخواست، انتخاب مدل و کشینگ کمک می‌کند تا هزینه‌ها تحت کنترل باقی بمانند بدون اینکه کیفیت قربانی شود.


## ساخت یک عامل قابل مشاهده

ما ابزارهای سفر را تعریف می‌کنیم و تماس با عامل را با زمان‌بندی می‌پوشانیم تا بتوانیم تأخیر را نظارت کنیم. در محیط تولید، شما با OpenTelemetry یا یک سیستم مشابه ردیابی ادغام می‌کنید.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## الگوهای ارزیابی

یک الگوی رایج تولید استفاده از یک عامل دوم به عنوان **ارزیاب** است. ارزیاب پاسخ عامل اصلی را بر اساس معیارهای از پیش تعیین شده مانند کامل بودن، دقت و مفید بودن امتیاز می‌دهد.

این امکان را فراهم می‌کند:
- دروازه‌های کیفیت خودکار قبل از رسیدن پاسخ‌ها به کاربران
- شناسایی پسرفت زمانی که پرامپت‌ها یا مدل‌ها تغییر می‌کنند
- پایش مستمر عملکرد عامل در طول زمان


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتژی‌های مدیریت هزینه

کنترل هزینه‌ها برای عوامل تولید مبتنی بر هوش مصنوعی امری حیاتی است. در اینجا استراتژی‌های کلیدی آورده شده است:

| استراتژی | توضیحات |
|---|---|
| **بهینه‌سازی پرامپت** | دستورالعمل‌های سیستم را مختصر نگه دارید. برای کاهش توکن‌های ورودی، زمینه‌های اضافی را حذف کنید. |
    "| **انتخاب مدل** | از مدل‌های کوچک‌تر و ارزان‌تر (مانند GPT-5-mini) برای وظایف ساده مانند دسته‌بندی یا استخراج استفاده کنید و مدل‌های بزرگ‌تر را فقط برای استدلال‌های پیچیده به کار ببرید. |\n",
| **کشینگ** | نتایج ابزارها و پرسش‌های تکراری را کش کنید تا از تماس‌های زائد API جلوگیری شود. |
| **بودجه توکن** | محدودیت `max_tokens` را تنظیم کنید تا پاسخ‌های طولانی غیرمنتظره جلوگیری شود. |
| **دسته‌بندی (batching)** | در صورت امکان، چندین پرسش کاربر را در یک تماس API گروه‌بندی کنید. |

در عمل، یک رویکرد چند سطحی مؤثر است: درخواست‌های ساده را به مدل سریع و ارزان هدایت کنید و تنها پرسش‌های پیچیده را به مدل قوی‌تر ارجاع دهید.


## خلاصه

در این درس یاد گرفتید چگونه:

1. **قابلیت مشاهده** را به تعاملات عامل با زمان‌بندی و ثبت اضافه کنید، زمینه‌ای برای ردیابی و نظارت فراهم کنید.
2. **پاسخ‌های عامل** را به صورت خودکار با استفاده از یک عامل ارزیاب که کامل بودن، دقت و مفید بودن را امتیاز می‌دهد، ارزیابی کنید.
3. **هزینه‌ها را مدیریت کنید** از طریق بهینه‌سازی درخواست، انتخاب مدل، کشینگ و بودجه‌بندی توکن.

این الگوهای تولید به شما کمک می‌کنند تا عامل‌های هوش مصنوعی شما در مقیاس، قابل اطمینان، قابل اندازه‌گیری و مقرون‌به‌صرفه باشند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
